# 04 — The Transformer Block

A single transformer block is the fundamental repeating unit in every transformer model. It combines multi-head attention with a feed-forward network, connected via residual connections and layer normalisation.

## Table of Contents
1. [Block Architecture Overview](#1-block-architecture-overview)
2. [Residual Connections](#2-residual-connections)
3. [Layer Normalisation](#3-layer-normalisation)
4. [Feed-Forward Network](#4-feed-forward-network)
5. [Building the Complete Block](#5-building-the-complete-block)
6. [Stacking Multiple Blocks](#6-stacking-multiple-blocks)
7. [Key Takeaways](#7-key-takeaways)

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import sys, os

sys.path.insert(0, os.path.abspath('.'))

torch.manual_seed(42)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Block Architecture Overview

A GPT-style transformer block (decoder-only, **pre-norm** variant):

```
Input x
    │
    ├──────────────────────────┐
    │                          │ (residual)
    ▼                          │
LayerNorm                      │
    │                          │
    ▼                          │
Multi-Head Self-Attention      │
(with causal mask)             │
    │                          │
    ▼                          │
  + ◄──────────────────────────┘   ← Add (residual connection)
    │
    ├──────────────────────────┐
    │                          │ (residual)
    ▼                          │
LayerNorm                      │
    │                          │
    ▼                          │
Feed-Forward Network           │
(expand → GELU → contract)     │
    │                          │
    ▼                          │
  + ◄──────────────────────────┘   ← Add (residual connection)
    │
    ▼
Output
```

**Pre-norm** means LayerNorm is applied **before** each sub-layer (GPT-2 style). This is more stable during training than the original post-norm variant.

Let's build each component.

## 2. Residual Connections

The residual (skip) connection adds the input directly to the output of a sub-layer:

$$\text{output} = x + \text{SubLayer}(x)$$

### Why?

- **Gradient flow**: Gradients can flow directly through the addition, avoiding vanishing gradients in deep networks
- **Identity fallback**: If the sub-layer learns nothing useful, the block just passes the input through
- **Incremental learning**: Each block learns a small **delta** (change) to add on top of the existing representation

In [ ]:
# ============================================================
# Demonstrate residual connections
# ============================================================

x = torch.tensor([1.0, 2.0, 3.0])  # input

# Imagine the sub-layer produces a small modification
sublayer_output = torch.tensor([0.1, -0.2, 0.3])  # learned delta

# Residual connection: add input back
output = x + sublayer_output

print("Residual Connection:")
print(f"  x (input):          {x.numpy()}")
print(f"  SubLayer(x) (delta): {sublayer_output.numpy()}")
print(f"  x + SubLayer(x):     {output.numpy()}")

print("\n" + "="*60)
print("KEY INSIGHT:")
print("="*60)
print("The sub-layer only needs to learn what to CHANGE (the residual),")
print("not reconstruct the entire representation from scratch.")
print("This makes training much easier and enables very deep networks.")
print("="*60)

## 3. Layer Normalisation

Layer normalisation stabilises training by normalising the activations **across the feature dimension** for each token independently.

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

Where:
- $\mu, \sigma^2$ are the mean and variance computed over the feature dimension (per token)
- $\gamma, \beta$ are learned scale and shift parameters
- $\epsilon$ is a small constant for numerical stability

In [ ]:
# ============================================================
# Layer Normalisation from scratch
# ============================================================

def layer_norm_manual(x, gamma, beta, eps=1e-5):
    """
    Manual layer normalisation.
    
    x:     (batch, seq, d_model)
    gamma: (d_model,)  — learned scale
    beta:  (d_model,)  — learned shift
    
    Normalises over the LAST dimension (d_model) independently for each
    token in each batch. This is different from batch norm, which
    normalises over the batch dimension.
    """
    # Mean and variance over the feature dimension (last dim)
    mean = x.mean(dim=-1, keepdim=True)       # (batch, seq, 1)
    var = x.var(dim=-1, keepdim=True, unbiased=False)  # (batch, seq, 1)
    
    # Normalise
    x_norm = (x - mean) / torch.sqrt(var + eps)  # (batch, seq, d_model)
    
    # Scale and shift with learned parameters
    out = gamma * x_norm + beta  # (batch, seq, d_model)
    
    return out

# Test with a small example
d_model = 8
x_test = torch.randn(1, 3, d_model)  # batch=1, seq=3, d=8

# Learned parameters (initialised to scale=1, shift=0)
gamma = torch.ones(d_model)
beta = torch.zeros(d_model)

# Our manual implementation
out_manual = layer_norm_manual(x_test, gamma, beta)

# PyTorch's implementation
ln = nn.LayerNorm(d_model)
out_pytorch = ln(x_test)

# Verify they match (with default gamma=1, beta=0)
print("Manual LayerNorm (token 0):")
print(f"  {out_manual[0, 0].detach().numpy().round(4)}")
print(f"  Mean: {out_manual[0, 0].mean().item():.6f}  (should be ≈ 0)")
print(f"  Std:  {out_manual[0, 0].std().item():.6f}  (should be ≈ 1)")

print("\nPyTorch LayerNorm (token 0):")
print(f"  {out_pytorch[0, 0].detach().numpy().round(4)}")

print(f"\nMatch: {torch.allclose(out_manual, out_pytorch, atol=1e-5)}")

In [ ]:
# ============================================================
# Visualise: Before vs after LayerNorm
# ============================================================

# Create input with varied scales across tokens
x_varied = torch.tensor([
    [[10.0, 20.0, 30.0, 40.0],    # token 0: large values
     [0.01, 0.02, 0.03, 0.04],    # token 1: tiny values
     [-5.0, 5.0, -5.0, 5.0]],     # token 2: mixed
])

ln4 = nn.LayerNorm(4)
x_normed = ln4(x_varied)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

labels = ['Token 0\n(large)', 'Token 1\n(tiny)', 'Token 2\n(mixed)']

# Before
for i in range(3):
    ax1.bar(np.arange(4) + i*4.5, x_varied[0, i].numpy(), label=labels[i])
ax1.set_title('Before LayerNorm', fontweight='bold')
ax1.set_ylabel('Value')
ax1.legend(fontsize=8)
ax1.axhline(y=0, color='grey', linewidth=0.5)

# After
for i in range(3):
    ax2.bar(np.arange(4) + i*4.5, x_normed[0, i].detach().numpy(), label=labels[i])
ax2.set_title('After LayerNorm', fontweight='bold')
ax2.set_ylabel('Value')
ax2.legend(fontsize=8)
ax2.axhline(y=0, color='grey', linewidth=0.5)

plt.suptitle('LayerNorm normalises each token independently to mean≈0, std≈1',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("After LayerNorm, all tokens have similar scale — regardless of their")
print("original magnitude. This stabilises gradient flow during training.")

## 4. Feed-Forward Network

The position-wise feed-forward network (FFN) is a simple two-layer MLP applied **independently** to each token position:

$$\text{FFN}(x) = \text{GELU}(x W_1 + b_1) W_2 + b_2$$

- **Expand**: $d_{model} \rightarrow d_{ff}$ (typically $d_{ff} = 4 \times d_{model}$)
- **GELU activation**: Smooth nonlinearity (used in GPT-2 instead of ReLU)
- **Contract**: $d_{ff} \rightarrow d_{model}$

### Why?

Attention handles token-to-token interactions. The FFN handles **per-token transformations** — like a lookup table that processes each token's representation independently.

In [ ]:
# ============================================================
# Feed-Forward Network from scratch
# ============================================================

class FeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network.
    
    Data flow:
        x (d_model) → Linear → GELU → Linear → output (d_model)
             ↓           ↓        ↓       ↓         ↓
        (batch,seq,64) → (batch,seq,256) → (batch,seq,64)
    
    The expansion to d_ff gives the network more capacity to learn
    complex per-token transformations.
    """
    
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)    # expand
        self.linear2 = nn.Linear(d_ff, d_model)    # contract
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch, seq, d_model)
        x = self.linear1(x)    # → (batch, seq, d_ff)
        x = F.gelu(x)          # smooth nonlinearity
        x = self.dropout(x)
        x = self.linear2(x)    # → (batch, seq, d_model)
        return x

# Test
d_model, d_ff = 64, 256
ff = FeedForward(d_model, d_ff)

x_test = torch.randn(1, 6, d_model)  # (batch=1, seq=6, d=64)
out_ff = ff(x_test)

print(f"FFN input:  {x_test.shape}  — (batch, seq, d_model={d_model})")
print(f"FFN output: {out_ff.shape}  — (batch, seq, d_model={d_model})")
print(f"\nInternally: {d_model} → {d_ff} → {d_model}  (4× expansion)")

# Parameter count
n_params = sum(p.numel() for p in ff.parameters())
print(f"\nFFN parameters: {n_params:,}")
print(f"  linear1: {d_model}×{d_ff} + {d_ff} = {d_model*d_ff + d_ff:,}")
print(f"  linear2: {d_ff}×{d_model} + {d_model} = {d_ff*d_model + d_model:,}")

In [ ]:
# ============================================================
# GELU vs ReLU comparison
# ============================================================

x_plot = torch.linspace(-4, 4, 200)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_plot.numpy(), F.relu(x_plot).numpy(), label='ReLU', linewidth=2)
ax.plot(x_plot.numpy(), F.gelu(x_plot).numpy(), label='GELU', linewidth=2)
ax.set_title('GELU vs ReLU Activation Functions', fontweight='bold')
ax.set_xlabel('Input')
ax.set_ylabel('Output')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='grey', linewidth=0.5)
ax.axvline(x=0, color='grey', linewidth=0.5)
plt.tight_layout()
plt.show()

print("GELU (Gaussian Error Linear Unit) is smoother than ReLU near x=0.")
print("It allows small negative values through, which helps gradient flow.")
print("GPT-2 and most modern transformers use GELU instead of ReLU.")

## 5. Building the Complete Block

Now we combine everything: Multi-Head Attention + FFN + LayerNorm + Residuals.

In [ ]:
# ============================================================
# First, bring in our MultiHeadAttention from notebook 03
# ============================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.last_attn_weights = None
    
    def forward(self, x, mask=None):
        batch, seq, _ = x.shape
        Q = self.W_q(x).view(batch, seq, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch, seq, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch, seq, self.n_heads, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = torch.nan_to_num(attn_weights, nan=0.0)
        self.last_attn_weights = attn_weights.detach()
        
        out = torch.matmul(self.dropout(attn_weights), V)
        out = out.transpose(1, 2).contiguous().view(batch, seq, self.d_model)
        return self.W_o(out)

In [ ]:
# ============================================================
# The complete Transformer Block
# ============================================================

class TransformerBlock(nn.Module):
    """
    A single GPT-style transformer decoder block (pre-norm).
    
    Data flow:
        x → LN → MHA → + x  (residual)
          → LN → FFN → + x  (residual)
    
    Args:
        d_model:  embedding dimension
        n_heads:  number of attention heads
        d_ff:     feed-forward inner dimension
        dropout:  dropout rate
    
    Input/Output: (batch, seq, d_model)
    """
    
    def __init__(self, d_model, n_heads, d_ff, dropout=0.0):
        super().__init__()
        
        # Layer normalisation (pre-norm: applied BEFORE each sub-layer)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
        # Multi-head self-attention
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        
        # Feed-forward network
        self.ff = FeedForward(d_model, d_ff, dropout)
        
        # Dropout for residual connections
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        """
        Forward pass through the block.
        
        x:    (batch, seq, d_model)
        mask: (1, 1, seq, seq) causal mask
        """
        # ---- Sub-layer 1: Multi-Head Self-Attention ----
        # Pre-norm: normalise before attention
        normed = self.ln1(x)                      # (batch, seq, d_model)
        attn_out = self.attn(normed, mask=mask)    # (batch, seq, d_model)
        x = x + self.dropout(attn_out)             # Residual connection
        
        # ---- Sub-layer 2: Feed-Forward Network ----
        # Pre-norm: normalise before FFN
        normed = self.ln2(x)                       # (batch, seq, d_model)
        ff_out = self.ff(normed)                    # (batch, seq, d_model)
        x = x + self.dropout(ff_out)               # Residual connection
        
        return x

# Test the complete block
d_model = 64
n_heads = 4
d_ff = 256
seq_len = 8

block = TransformerBlock(d_model, n_heads, d_ff)
x_test = torch.randn(1, seq_len, d_model)

# Causal mask
mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
mask = mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq, seq)

out = block(x_test, mask=mask)

print(f"TransformerBlock:")
print(f"  Input:  {x_test.shape}")
print(f"  Output: {out.shape}")
print(f"  Shape preserved: {x_test.shape == out.shape}  ✓")

# Parameter count
n_params = sum(p.numel() for p in block.parameters())
print(f"\n  Total parameters: {n_params:,}")

In [ ]:
# ============================================================
# Trace the data flow through the block with shape annotations
# ============================================================

print("="*70)
print("DATA FLOW THROUGH A TRANSFORMER BLOCK")
print("="*70)

x = torch.randn(1, seq_len, d_model)
print(f"\nInput x:                           {list(x.shape)}")

# Sub-layer 1: Attention
normed1 = block.ln1(x)
print(f"After LayerNorm 1:                 {list(normed1.shape)}")

attn_out = block.attn(normed1, mask=mask)
print(f"After Multi-Head Attention:        {list(attn_out.shape)}")

x = x + attn_out  # residual
print(f"After residual connection (+ x):   {list(x.shape)}")

# Sub-layer 2: FFN
normed2 = block.ln2(x)
print(f"After LayerNorm 2:                 {list(normed2.shape)}")

ff_internal = block.ff.linear1(normed2)
print(f"FFN expand (d_model → d_ff):       {list(ff_internal.shape)}")

ff_activated = F.gelu(ff_internal)
print(f"After GELU:                        {list(ff_activated.shape)}")

ff_out = block.ff.linear2(ff_activated)
print(f"FFN contract (d_ff → d_model):     {list(ff_out.shape)}")

x = x + ff_out  # residual
print(f"After residual connection (+ x):   {list(x.shape)}")

print(f"\nFinal output:                      {list(x.shape)}")
print(f"\n✓ Shape is preserved throughout: (batch, seq, d_model)")

## 6. Stacking Multiple Blocks

A full transformer stacks multiple blocks. Each block refines the representations further — earlier blocks capture simpler patterns, later blocks capture more complex, abstract relationships.

In [ ]:
# ============================================================
# Stack multiple transformer blocks
# ============================================================

n_layers = 4

blocks = nn.ModuleList([
    TransformerBlock(d_model=64, n_heads=4, d_ff=256)
    for _ in range(n_layers)
])

# Run input through all blocks sequentially
x = torch.randn(1, 8, 64)  # (batch=1, seq=8, d=64)
mask = torch.triu(torch.ones(8, 8, dtype=torch.bool), diagonal=1).unsqueeze(0).unsqueeze(0)

print(f"Stacking {n_layers} transformer blocks:")
print(f"\n  Input: {list(x.shape)}")

for i, block in enumerate(blocks):
    x = block(x, mask=mask)
    print(f"  After block {i}: {list(x.shape)}")

print(f"\n  Final: {list(x.shape)}")

# Total parameters
total_params = sum(p.numel() for p in blocks.parameters())
per_block = total_params // n_layers
print(f"\n  Parameters per block: {per_block:,}")
print(f"  Total parameters ({n_layers} blocks): {total_params:,}")

print("\n" + "="*60)
print("KEY INSIGHT:")
print("="*60)
print("Each block's output has the same shape as its input.")
print("This means blocks are perfectly stackable — you can add")
print("as many as you want (limited only by compute and memory).")
print("")
print("GPT-2 Small:  12 blocks, d_model=768,  ~117M parameters")
print("GPT-2 Large:  36 blocks, d_model=1280, ~774M parameters")
print("GPT-3:        96 blocks, d_model=12288, ~175B parameters")
print("="*60)

In [ ]:
# ============================================================
# Verify: Residual connections prevent vanishing activations
# ============================================================

# Compare activation magnitudes with and without residuals
n_test_layers = 20
x_with_res = torch.randn(1, 4, 64)
x_without_res = x_with_res.clone()

norms_with = [x_with_res.norm().item()]
norms_without = [x_without_res.norm().item()]

for _ in range(n_test_layers):
    # With residual
    block_out = TransformerBlock(64, 4, 256)(x_with_res, mask=mask[:,:,:4,:4])
    x_with_res = block_out  # block already has residuals
    norms_with.append(x_with_res.norm().item())
    
    # Without residual (just FFN, no skip connection)
    ff_only = FeedForward(64, 256)
    x_without_res = ff_only(x_without_res)  # no residual!
    norms_without.append(x_without_res.norm().item())

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(norms_with, 'o-', label='With residuals (TransformerBlock)', linewidth=2)
ax.plot(norms_without, 's--', label='Without residuals (FFN only)', linewidth=2)
ax.set_xlabel('Layer', fontsize=11)
ax.set_ylabel('Activation Norm', fontsize=11)
ax.set_title('Residual Connections Maintain Activation Magnitude', fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Key Takeaways

---

### Summary

| Component | Purpose |
|-----------|--------|
| **Multi-Head Attention** | Token-to-token interactions — gathers context from other positions |
| **Feed-Forward Network** | Per-token transformation — processes each position independently |
| **Layer Normalisation** | Stabilises activations — keeps training well-behaved |
| **Residual Connections** | Enables gradient flow — makes deep stacking possible |
| **Pre-norm ordering** | LN before sub-layer — more stable than post-norm (GPT-2 style) |
| **GELU activation** | Smoother than ReLU — better gradient properties for transformers |

### Architecture at a Glance

```
x → LN → MHA → +x → LN → FFN → +x → output
```

This pattern repeats N times (one block per layer). The output of each block is the same shape as the input, so they're perfectly composable.

### What's Next?

In the next notebook (**05 — Training**), we put everything together into a full GPT-style language model, train it on text, and generate new text from it.

---